# UmarTransit-1B: Model Evaluation

Evaluate the fine-tuned UmarTransit-1B model on 335 test pairs.

**Metrics:** ROUGE-L (content overlap), keyword matching (factual accuracy)
**Breakdown:** Per-category analysis across 8 task categories

> **Requirements:** Google Colab with T4 GPU runtime.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers torch accelerate rouge-score nltk tqdm

## 2. Configuration

In [ ]:
MODEL_ID = "umarfarookm/UmarTransit-1B"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.9

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/umarfarookm/transit-foundation-model/main"
TEST_URL = f"{GITHUB_RAW_BASE}/datasets/synthetic/test.jsonl"

SYSTEM_PROMPT = (
    "You are UmarTransit-1B, a specialized AI assistant for public transit systems "
    "and GTFS (General Transit Feed Specification) data. You provide accurate, "
    "detailed answers about transit routes, stops, schedules, transfers, and GTFS concepts."
)

## 3. Load Model & Test Data

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading model from HuggingFace...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Model loaded: {MODEL_ID}")
print(f"Device: {model.device}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
import json
import requests

def load_jsonl_from_url(url):
    response = requests.get(url)
    response.raise_for_status()
    return [json.loads(line) for line in response.text.strip().split("\n") if line.strip()]

print("Loading test dataset from GitHub...")
test_data = load_jsonl_from_url(TEST_URL)
print(f"Loaded {len(test_data)} test pairs")

# Category distribution
from collections import Counter
cats = Counter(d["category"] for d in test_data)
print("\nCategory distribution:")
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count}")

## 4. Run Inference on All Test Pairs

Generate responses for all 335 test questions (~3-5 min on T4).

In [ ]:
from tqdm import tqdm

def generate_response(question):
    """Generate a response from the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

# Run inference on all test pairs
results = []
for item in tqdm(test_data, desc="Evaluating"):
    generated = generate_response(item["instruction"])
    results.append({
        "instruction": item["instruction"],
        "expected": item["response"],
        "generated": generated,
        "category": item["category"],
        "template_id": item.get("template_id", ""),
        "feed_id": item.get("feed_id", ""),
    })

print(f"\nGenerated {len(results)} responses")

## 5. Compute Metrics

In [ ]:
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from rouge_score import rouge_scorer
from collections import defaultdict

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def extract_numbers(text):
    """Extract all numbers from text for keyword matching."""
    return set(re.findall(r'\b\d[\d,]*\.?\d*\b', text.replace(',', '')))

def keyword_match_score(expected, generated):
    """Check what fraction of key numbers from expected appear in generated."""
    expected_nums = extract_numbers(expected)
    if not expected_nums:
        return 1.0  # No numbers to check
    generated_nums = extract_numbers(generated)
    matched = expected_nums & generated_nums
    return len(matched) / len(expected_nums)

# Compute metrics for each result
rouge_scores = []
keyword_scores = []
category_metrics = defaultdict(lambda: {"rouge": [], "keyword": [], "count": 0})

for r in results:
    # ROUGE-L
    score = scorer.score(r["expected"], r["generated"])
    rouge_l = score["rougeL"].fmeasure
    rouge_scores.append(rouge_l)
    r["rouge_l"] = rouge_l

    # Keyword match
    kw = keyword_match_score(r["expected"], r["generated"])
    keyword_scores.append(kw)
    r["keyword_match"] = kw

    # Per-category
    cat = r["category"]
    category_metrics[cat]["rouge"].append(rouge_l)
    category_metrics[cat]["keyword"].append(kw)
    category_metrics[cat]["count"] += 1

# Overall metrics
avg_rouge = sum(rouge_scores) / len(rouge_scores)
avg_keyword = sum(keyword_scores) / len(keyword_scores)

print("=" * 60)
print("OVERALL METRICS")
print("=" * 60)
print(f"  ROUGE-L (avg):        {avg_rouge:.4f}")
print(f"  Keyword Match (avg):  {avg_keyword:.4f}")
print(f"  Total test pairs:     {len(results)}")

## 6. Per-Category Analysis

In [ ]:
print("=" * 70)
print(f'{"Category":<25} {"Count":>6} {"ROUGE-L":>10} {"Keyword":>10}')
print("-" * 70)

for cat in sorted(category_metrics.keys()):
    m = category_metrics[cat]
    avg_r = sum(m["rouge"]) / len(m["rouge"])
    avg_k = sum(m["keyword"]) / len(m["keyword"])
    print(f'{cat:<25} {m["count"]:>6} {avg_r:>10.4f} {avg_k:>10.4f}')

print("-" * 70)
print(f'{"OVERALL":<25} {len(results):>6} {avg_rouge:>10.4f} {avg_keyword:>10.4f}')

# Show best and worst examples per category
print("\n" + "=" * 70)
print("EXAMPLES: BEST AND WORST PER CATEGORY")
print("=" * 70)

for cat in sorted(category_metrics.keys()):
    cat_results = [r for r in results if r["category"] == cat]
    cat_results.sort(key=lambda r: r["rouge_l"], reverse=True)

    best = cat_results[0]
    worst = cat_results[-1]

    print(f"\n--- {cat} (best ROUGE-L: {best['rouge_l']:.3f}) ---")
    print(f"  Q: {best['instruction'][:100]}")
    print(f"  Expected:  {best['expected'][:120]}")
    print(f"  Generated: {best['generated'][:120]}")

    print(f"\n--- {cat} (worst ROUGE-L: {worst['rouge_l']:.3f}) ---")
    print(f"  Q: {worst['instruction'][:100]}")
    print(f"  Expected:  {worst['expected'][:120]}")
    print(f"  Generated: {worst['generated'][:120]}")

## 7. Save Evaluation Report

In [ ]:
from datetime import datetime, timezone

# Build report
category_summary = {}
for cat in sorted(category_metrics.keys()):
    m = category_metrics[cat]
    category_summary[cat] = {
        "count": m["count"],
        "rouge_l_avg": round(sum(m["rouge"]) / len(m["rouge"]), 4),
        "keyword_match_avg": round(sum(m["keyword"]) / len(m["keyword"]), 4),
    }

report = {
    "model_id": MODEL_ID,
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "test_pairs": len(results),
    "overall_metrics": {
        "rouge_l_avg": round(avg_rouge, 4),
        "keyword_match_avg": round(avg_keyword, 4),
    },
    "category_metrics": category_summary,
    "predictions": results,
}

# Save to Colab filesystem
report_path = "evaluation_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Evaluation report saved to: {report_path}")
print(f"\nDownload it from Colab's file browser (left panel).")

# Final summary
print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)
print(f"  Model:           {MODEL_ID}")
print(f"  Test pairs:      {len(results)}")
print(f"  ROUGE-L:         {avg_rouge:.4f}")
print(f"  Keyword Match:   {avg_keyword:.4f}")
print(f"\nStrengths:")
best_cat = max(category_summary, key=lambda c: category_summary[c]['rouge_l_avg'])
print(f"  Best category: {best_cat} (ROUGE-L: {category_summary[best_cat]['rouge_l_avg']})")
print(f"\nAreas for improvement:")
worst_cat = min(category_summary, key=lambda c: category_summary[c]['rouge_l_avg'])
print(f"  Weakest category: {worst_cat} (ROUGE-L: {category_summary[worst_cat]['rouge_l_avg']})")